#### BROKEN NOTEBOOK - CURRENT WORKFLOW NEEDS ATTENTION

# Intersect Community Data Workflow for Housing Unit Allocation with Person Record Files with Disability

# This notebook includes Full Housing Unit Allocation workflow with Person Record File with optional Disability Status

## Overview
This code runs the HUA and PREC workflows and then adds the Housing Unit ID to the PREC file. 

## Required Inputs
Program requires the following inputs:

[Census API KEY *REQUIRED*](CENSUS_API_KEY.md) See CENSUS_API_KEY.md file for more details.
    
## Output Description
The output of this workflow is a CSV file with the person record file with disability status.

## Instructions
Users can run the workflow by executing each block of code in the notebook.

## Description of Program
- program:    ncoda_07lv1_HUA_PREC
- task:       intersect HUA and PREC 
- See github commits for description of program updates
- Current Version: v1 - 
- 2026-09-01 - Integrate disability work into PREC workflow
- project:    Texas Mitigation Planning Initiative
- funding:	  FEMA
- author:     Nathanael Rosenheim, Emmanuel Randle and Swastika Barua

## Required Citations:
Rosenheim, Nathanael, Roberto Guidotti, Paolo Gardoni & Walter Gillis Peacock. (2021). Integration of detailed household and housing unit characteristic data with critical infrastructure for post-hazard resilience modeling. _Sustainable and Resilient Infrastructure_. 6(6), 385-401. https://doi.org/10.1080/23789689.2019.1681821

Rosenheim, Nathanael (2021) “Detailed Household and Housing Unit Characteristics: Data and Replication Code.” _DesignSafe-CI_. 
https://doi.org/10.17603/ds2-jwf6-s535.

In [1]:
# To reload submodules need to use this magic command to set autoreload on
%load_ext autoreload
%autoreload 2
from pyncoda.ncoda_00g_community_options import *
from IPython.display import display

### How to set up the Community Dictionary
Please review the python code in the file pyncoda/ncoda_00g_community_options.py

In this file you will find a collection of data dictionaries with various ways to setup the inputs for the Housing Unit Allocation process. 

The basic dictionary includes the name of the community, the county FIPS code, your input building inventory file, and key variables in the building inventory file.

In [2]:
# select a community from this list
# if your community is not in this list, add it to the file ncoda_00g_community_options.py
list_community_options(communities_dictionary)

['Lumberton, NC: IN-CORE Building inventory for Robeson County, NC',
 'Galveston, TX: IN-CORE Building inventory for Galveston County, TX',
 'Galveston, TX: NSI Building inventory for Galveston County, TX',
 'Galveston, TX: IN-CORE Building inventory for Galveston Island, TX',
 'Mayfield, KY: NSI Building inventory for Graves County, KY',
 'Beaumont, TX: NSI Building inventory for Jefferson County, TX',
 'Beaumont, TX: Safayet Building inventory for Jefferson County, TX',
 'Pentwater, MI: NSI Building inventory for Oceana County, MI',
 'Seaside, OR: NSI Building inventory for Clatsop County, OR',
 'Lane County, OR: NSI Building inventory for Lane County, OR',
 'Benton County, OR: NSI Building inventory for Benton County, OR',
 'Southeast Texas Urban Integrated Field Lab: NSI Building inventory for Southeast Texas',
 'Southeast Texas Urban Integrated Field Lab (12 neighbor counties): NSI Building inventory for Southeast Texas',
 'Brazos County, TX: NSI Building inventory for Brazos Coun

In [3]:
community_id_by_name =  'Seaside, OR: NSI Building inventory for Clatsop County, OR'

In [4]:
community_id, focalplace, countyname, countyfips = get_community_id_by_name(community_id_by_name)
communities = {community_id : communities_dictionary[community_id]}

Selected community ID: Seaside_OR_NSI
Seaside, OR is in OREGON
Focal place: Seaside
Seaside, OR is in Clatsop County, OR with FIPS code 41007
Use IN-CORE: False


## Setup Python Environment

In [5]:
import pandas as pd
import geopandas as gpd # For reading in shapefiles
import numpy as np
import sys # For displaying package versions
import os # For managing directories and file paths if drive is mounted
import scooby # Reports Python environment

import contextily as cx # For adding basemap tiles to plot
import matplotlib.pyplot as plt # For plotting and making graphs

In [6]:
# open, read, and execute python program with reusable commands
from pyncoda.ncoda_00d_cleanvarsutils import *
from pyncoda.ncoda_04c_poptableresults import *
from pyncoda.ncoda_07i_process_communities import process_community_workflow

In [7]:
# Generate report of Python environment
base_packages = ['pandas','ipyleaflet','seaborn','contextily']
incore_packages = ['pyincore','pyincore_viz']
check_packages = base_packages + incore_packages
print(scooby.Report(additional=check_packages))


--------------------------------------------------------------------------------
  Date: Tue Sep 01 16:29:57 2026 Eastern Daylight Time

                OS : Windows (10 10.0.26200 SP0 Multiprocessor Free)
            CPU(s) : 16
           Machine : AMD64
      Architecture : 64bit
               RAM : 31.7 GiB
       Environment : Jupyter

  Python 3.10.14 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:44:50)
  [MSC v.1916 64 bit (AMD64)]

            pandas : 2.2.2
        ipyleaflet : Module not found
           seaborn : 0.13.2
        contextily : 1.6.0
          pyincore : Module not found
      pyincore_viz : Module not found
             numpy : 1.26.4
             scipy : 1.13.1
           IPython : 8.25.0
        matplotlib : 3.8.4
            scooby : 0.10.0

  Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303
  for Intel(R) 64 architecture applications
--------------------------------------------------------------------------------


In [8]:
# Check working directory - good practice for relative path access
os.getcwd()

'c:\\Users\\nathanael99\\MyProjects\\GitHub\\intersect-community-data'

## Run Housing Unit Allocation
The following code will produce the following outputs:
1. Housing Unit Inventory
2. Address Point Inventory
3. Housing Unit Allocation

In [9]:
basevintage_options = ['2010','2020']

In [10]:
hua_hui_gdf_dict = {}
base_seed = 9876
iterations = 1
# iterate through basevintage options to run Monte Carlo Simulation
for basevintage in basevintage_options:
    hua_hui_gdf_dict[basevintage] = {}
    for i in range(iterations):
        seed_i = base_seed + i
        print(f"Running iteration {i+1} of {iterations} for basevintage {basevintage} with seed {seed_i}")
        workflow = process_community_workflow(
                    communities,
                    seed = seed_i,
                    version = '2.2.0',
                    version_text = 'v2-2-0',
                    basevintage = basevintage,
                    outputfolder ="OutputData",
                    outputfolders = {},
                    savefiles = True)
        hua_hui_gdf_dict[basevintage][seed_i] = workflow.process_communities()

Running iteration 1 of 1 for basevintage 2010 with seed 9876
Generating Housing Unit Inventory v2-2-0 data for Seaside, OR
Clatsop County, OR : county FIPS Code 41007
File already exists, skipping: OutputData/Seaside_OR_NSI/../hui_v2-2-0_Seaside_OR_NSI_2010_rs9876.csv
Checking output for huid
Checking output for blockid
Checking output for bgid
Checking output for tractid
Checking output for FIPScounty
Checking output for numprec
Checking output for ownershp
Checking output for race
Checking output for hispan
Checking output for family
Checking output for vacancy
Checking output for gqtype
Checking output for incomegroup
Checking output for hhinc
Checking output for randincome
Checking output for poverty
Checking huid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
Checking blockid Data Type
   Current type: <class 'pandas.core.series.Series'> Expected type <class 'str'>
    Length of blockid is correct
Checking bgid Data Type
   Current type:

### Run Person Record File Generation

In [ ]:
version = '3.0.0'
version_text = 'v3-0-0'

# open, read, and execute python program with reusable commands
from pyncoda.ncoda_07e_generate_prec import generate_prec_functions

# Save Outputfolder - due to long folder name paths output saved to folder with shorter name
# files from this program will be saved with the program name - 
# this helps to follow the overall workflow
outputfolder = "OutputData"
# Make directory to save output
if not os.path.exists(outputfolder):
    os.mkdir(outputfolder)

# Set random seed for reproducibility
seed = 1000
basevintage = 2010

generate_prec_df = generate_prec_functions(
                    communities =   communities,
                    seed =          seed,
                    version =       version,
                    version_text=   version_text,
                    basevintage=    basevintage,
                    outputfolder=   outputfolder
                    )

prec_df = generate_prec_df.generate_prec_v300()

Generating Person Record File v3.0.0 data for Seaside, OR
Clatsop County, OR : county FIPS Code 41007

***************************************
    Version control - list of installed packages
***************************************

Unable to print version information

***************************************
    Obtain and clean core person record characteristics for Clatsop County, OR
***************************************

{'top': 'OutputData/Seaside_OR_NSI', 'logfiles': 'OutputData/Seaside_OR_NSI/00_logfiles', 'CommunitySourceData': 'OutputData/Seaside_OR_NSI/01_CommunitySourceData', 'TidyCommunitySourceData': 'OutputData/Seaside_OR_NSI/02_TidyCommunitySourceData', 'BaseInventory': 'OutputData/Seaside_OR_NSI/03_BaseInventory', 'RandomMerge': 'OutputData/Seaside_OR_NSI/04_RandomMerge', 'Verify': 'OutputData/Seaside_OR_NSI/05_Verify', 'Explore': 'OutputData/Seaside_OR_NSI/06_Explore', 'Uncertainty_propagation': 'OutputData/Seaside_OR_NSI/07_Uncertainty_propagation', 'Validation': 'Ou

KeyError: 'race'

## Intersect HUA and PREC (PRECHUI)

The Housing Unit Allocation loop above produced housing units and how many people live in
each. The Person Record cells produced people with age, sex, race, ethnicity and
disability. Neither says **which person lives in which housing unit**. This section joins
them, so a person record carries a `huid` - and, through the housing unit's allocation,
the building id (`fd_id_bid`) and point coordinates. (`strctid` and `addrptid` are
inherited too when the HUA file carries them; the current `process_communities` saves
only the building id, place name, `huestimate` and x/y.)

It runs in four steps:

1. **Householder characteristics** - the housing unit inventory carries race and
   ethnicity but not the age or sex of the householder, and the linkage joins on both.
   These come from tenure by age of householder (`H13` in 2020, `H17` in 2010) and tenure
   by household type by age of householder (`H14` / `H18`).
2. **Prepare housing units** - adjust households above the inventory's cap of 7, expand
   each unit into one row per resident, and clear the characteristics that expansion
   wrongly copied from the householder.
3. **Link** - place group quarters residents, then random merge person records onto the
   housing unit slots.
4. **Validate** - structural invariants, then a comparison against Census tables the
   merge was *not* fitted to.

New in this notebook: after the structural validation, each linked person inherits the
building attributes of their housing unit by one left join on `huid` against the full
HUA file of the same vintage and seed - the building id, `huestimate`, place name and
x/y point geometry, plus `strctid`/`addrptid` when the file carries them.

What this section needs from the cells above:

- the Housing Unit Allocation loop must have run with `savefiles = True` for the vintage
  being linked - it writes the pre-polish housing unit inventory and the full HUA file
  this section reads, both under the notebook's `OutputData` tree;
- the Person Record cell must have run, leaving `prec_df` in memory. The setup cell
  below refuses a person frame built for the
  wrong vintage and warns loudly on a mismatched seed, because linking mismatched
  ensembles silently pairs one run's people with another run's housing units;
- the `CENSUS_API_KEY` environment variable must be set - the householder tables, the
  group quarters table and the out-of-sample check all go through the Census API.

In [ ]:
# ==============================================================
# Setup: one vintage switch, one seed, separate output tree
# ==============================================================
import glob
import os

import geopandas as gpd
import pandas as pd
import requests

from pyncoda.CommunitySourceData.api_census_gov.acg_00a_createAPI_datastructure_2020_patch import (
    patch_obtain_api_metadata_for_2020,
    patch_get_data_for_2020,
)
from pyncoda.CommunitySourceData.api_census_gov.acg_05b_prec_functions import (
    prec_workflow_functions,
)
from pyncoda.CommunitySourceData.api_census_gov.acg_05c_hui_householder import (
    hui_householder_functions,
)
from pyncoda.ncoda_07f_run_prechui_workflow import prechui_workflow_functions

# The single vintage switch for the whole section. It follows the PREC cell's
# basevintage so the two frames always describe the same decade; the HUA loop
# above ran both vintages, so overriding to '2010' or '2020' also works.
intersect_vintage = str(basevintage)
assert intersect_vintage in basevintage_options, \
    f"intersect_vintage {intersect_vintage} is not in {basevintage_options}"

# One seed threads through everything: every on-disk input this section reads
# is keyed rs{base_seed}, and the linkage constructors reuse it so their cache
# filenames and merge randomness stay coherent. Deliberately NOT the bare
# `seed` global - the PREC cells leave that bound to their own value.
intersect_seed = base_seed

# The linkage modules speak state_county / state_county_name
state_county = countyfips
state_county_name = countyname

# Matches the Grays Harbor witness convention for prechui filenames. The HUA
# join below alone uses the HUA loop's own version text.
prechui_version_text = 'v2-1-0'

# --- Guards: refuse to link mismatched ensembles -----------------------------
try:
    prec_df
except NameError:
    raise NameError(
        "prec_df is not defined - run the Person Record Generation cell "
        "above first.") from None

if 'precid' not in prec_df.columns:
    raise ValueError(
        "prec_df has no precid column, so it is not the person record "
        "frame. Rerun the Person Record Generation cell above.")

# (a) prec_df must carry the block geography of the vintage being linked.
#     Without it the block pivots in adjust_numprec7_hui are meaningless.
if f'Block{intersect_vintage}str' not in prec_df.columns:
    raise ValueError(
        f"prec_df has no Block{intersect_vintage}str column - it was built "
        f"for a different vintage. Rerun the PREC cells above with "
        f"basevintage = {intersect_vintage}.")

# (b) Warn - without touching the PREC cells' variables - if the PREC run used
#     a different seed or vintage than this section links against.
if str(basevintage) != intersect_vintage or seed != intersect_seed:
    print("*" * 72)
    print(f"WARNING: the PREC cells ran with seed = {seed} and "
          f"basevintage = {basevintage},")
    print(f"but this section links with intersect_seed = {intersect_seed} "
          f"and intersect_vintage = {intersect_vintage}.")
    print("Edit the PREC setup cell to seed = base_seed and "
          "basevintage = intersect_vintage,")
    print("then rerun the PREC cells. Linking mismatched ensembles silently "
          "pairs one run's people")
    print("with another run's housing units.")
    print("*" * 72)

# (c) Every pull below goes through the Census API, and a keyless request
#     comes back as a 200 HTML page disguised as a JSONDecodeError - so check
#     up front, not at first failure.
assert os.environ.get('CENSUS_API_KEY'), \
    "CENSUS_API_KEY environment variable is not set - see CENSUS_API_KEY.md"

# (d) Nothing in the package installs the 2020 API patches automatically;
#     without them the 2020 householder pull (H13/H14) fails or mis-parses.
if intersect_vintage == '2020':
    patch_obtain_api_metadata_for_2020()
    patch_get_data_for_2020()

# --- Output tree: separate from the HUA tree so result caches never collide --
prechui_outputfolder = "OutputData_PRECHUI"
prechui_base = f"{prechui_outputfolder}/{community_id}"
prechui_outputfolders = {
    'top'                    : prechui_base,
    'logfiles'               : f"{prechui_base}/00_logfiles",
    'CommunitySourceData'    : f"{prechui_base}/01_CommunitySourceData",
    'TidyCommunitySourceData': f"{prechui_base}/02_TidyCommunitySourceData",
    'BaseInventory'          : f"{prechui_base}/03_BaseInventory",
    'RandomMerge'            : f"{prechui_base}/04_RandomMerge",
}
for folder in prechui_outputfolders.values():
    os.makedirs(folder, exist_ok=True)

# The random merge reloads a finished result pair from 04_RandomMerge and
# SKIPS the merge entirely if both files exist, regardless of code or input
# changes. After any upstream rerun with the SAME seed and vintage - or after
# changing geo_levels or the round ladder - flip this on once to defeat the
# cache.
prechui_force_rerun = False
if prechui_force_rerun:
    stale_pattern = (f"{prechui_outputfolders['RandomMerge']}/"
                     f"*_{state_county}_{intersect_vintage}"
                     f"_rs{intersect_seed}_*.csv")
    for stale_file in glob.glob(stale_pattern):
        print("Removing cached random merge result:", stale_file)
        os.remove(stale_file)

# --- Inputs ------------------------------------------------------------------
# Every HUA artifact lives under the HUA loop's own output tree - the
# outputfolder argument the Housing Unit Allocation loop above passes.
hua_outputfolder = "OutputData"

# The housing unit inventory BEFORE final polish, which still carries the
# block geography and householder keys the merge needs - the polished file
# drops Block{vintage}str. The HUA loop above wrote it under the community's
# 04_RandomMerge folder.
hui_path = (f"{hua_outputfolder}/{community_id}/04_RandomMerge/"
            f"hui_B19001rmB19101_{state_county}_{intersect_vintage}"
            f"_rs{intersect_seed}_primary.csv")
try:
    hui_df = pd.read_csv(hui_path, low_memory=False)
except FileNotFoundError:
    raise FileNotFoundError(
        f"{hui_path} does not exist - run the Housing Unit Allocation cell "
        f"above (savefiles=True) for basevintage {intersect_vintage} first."
    ) from None

print(f"Housing units : {len(hui_df):,}  "
      f"(numprec sum {int(hui_df['numprec'].sum()):,})")
print(f"Person records: {len(prec_df):,}")
print(f"Community     : {community_id}   seed {intersect_seed}   "
      f"vintage {intersect_vintage}")

In [ ]:
# ==============================================================
# Step 1: Add householder age and sex to the housing unit inventory
#
# The inventory has race and ethnicity but not the householder's age band or
# sex, which are two of the four keys the linkage joins on. The vintage picks
# the tables automatically: H17/H18 in 2010, H13/H14 in 2020.
# ==============================================================
householder = hui_householder_functions(
    state_county      = state_county,
    state_county_name = state_county_name,
    seed              = intersect_seed,
    version           = '2.1.0',
    version_text      = prechui_version_text,
    basevintage       = intersect_vintage,
    basegeolevel      = 'Block',
    outputfolder      = prechui_outputfolder,
    outputfolders     = prechui_outputfolders,
)
print("Vintage mapping:", {k: v for k, v in householder.tables.items()
                           if k in ('age_by_tenure', 'type_by_age', 'dataset_name')})

hui_householder_result = householder.add_householder_characteristics(hui_df)
hui_hh_df = (hui_householder_result['primary']
             if isinstance(hui_householder_result, dict) else hui_householder_result)

print(f"\nHousing units with householder characteristics: {len(hui_hh_df):,}")
for check, (passed, detail) in hui_householder_functions.\
        validate_householder_characteristics(hui_df, hui_hh_df).items():
    print(f"   [{'PASS' if passed else 'FAIL'}] {check:<46} {detail}")

In [ ]:
# ==============================================================
# Step 2: Prepare the housing units
#
# Households above the inventory cap of 7 are enlarged to absorb their
# block's person shortfall, each unit is expanded into one row per resident,
# and the characteristics that expansion wrongly copied to non-householders
# are cleared.
# ==============================================================
linkage = prechui_workflow_functions(
    state_county      = state_county,
    state_county_name = state_county_name,
    seed              = intersect_seed,
    version           = '2.1.0',
    version_text      = prechui_version_text,
    basevintage       = intersect_vintage,
    basegeolevel      = 'Block',
    outputfolder      = prechui_outputfolder,
    outputfolders     = prechui_outputfolders,
)

# verify_results stays False - True changes the return to a tuple
hui_adjusted = linkage.adjust_numprec7_hui(hui_df = hui_hh_df, prec_df = prec_df)
print(f"Persons implied before adjustment: {int(hui_hh_df['numprec'].sum()):,}")
print(f"Persons implied after  adjustment: {int(hui_adjusted['numprec'].sum()):,}")

hui_slots = linkage.infer_household_structure(
    linkage.expand_hui_to_persons(hui_adjusted))
print(f"\nPerson slots: {len(hui_slots):,}")

for check, (passed, detail) in linkage.validate_person_slots(
        hui_adjusted, hui_slots).items():
    print(f"   [{'PASS' if passed else 'FAIL'}] {check:<40} {detail}")

In [ ]:
# ==============================================================
# Step 3: Link person records to housing units
#
# Group quarters residents first - they have no householder, so they are
# matched through the group quarters table - then everyone else.
# ==============================================================
gq_worker = prec_workflow_functions(
    state_county      = state_county,
    state_county_name = state_county_name,
    seed              = intersect_seed,
    version           = '2.1.0',
    version_text      = prechui_version_text,
    # passed explicitly: the constructor defaults to '2010', and the silent
    # default would pull the wrong decade's group quarters table
    basevintage       = intersect_vintage,
    outputfolder      = prechui_outputfolder,
    outputfolders     = prechui_outputfolders,
)
# 'person' gives one row per group quarters RESIDENT. 'housingunit' returns
# one row per facility, which looks plausible and strands most GQ residents.
groupquarters_df = gq_worker.tidy_group_quarters(unit_of_analysis = 'person')
print(f"Group quarters residents: {len(groupquarters_df):,}")

gq_merged = linkage.merge_groupquarters(hui_slots, groupquarters_df)
hui_slots_gq = gq_merged['primary'] if isinstance(gq_merged, dict) else gq_merged

prec_ready = linkage.prepare_prec_for_merge(prec_df)

# Block-only keeps every placed person in the block they actually live in, at
# the cost of leaving some unplaced. If placement rate matters more than block
# fidelity, the escalation is linkage.merge_prec_to_hui_staged(...), which
# runs the age-preserving rounds at wider geographies before giving age up.
prechui_merged = linkage.merge_prec_to_hui(prec_ready, hui_slots_gq,
                                           geo_levels = ['Block'])
prechui_df = linkage.polish_prechui(prechui_merged['primary'])

# polish_prechui writes nothing itself; save the deliverable with the
# ensemble identity - community, vintage, seed - in the name
prechui_path = (f"{prechui_outputfolders['top']}/"
                f"prechui_{prechui_version_text}_{state_county}"
                f"_{intersect_vintage}_rs{intersect_seed}.csv")
prechui_df.to_csv(prechui_path, index = False)

unassigned = linkage.unassigned_mask(prechui_df['huid'])
print(f"\nLinked person records: {len(prechui_df):,}")
print(f"   assigned a huid   : {int((~unassigned).sum()):,} "
      f"({(~unassigned).sum() / len(prechui_df) * 100:.2f}%)")
print(f"   unassigned        : {int(unassigned.sum()):,}")
print(f"Saved: {prechui_path}")

In [ ]:
# ==============================================================
# Step 4: Structural validation
#
# What a linkage can get wrong: losing or duplicating people, overfilling a
# housing unit, inventing a huid, or moving someone out of their own block.
# A check that selects nothing fails rather than passes.
# ==============================================================
linkage_checks = linkage.validate_linkage(prec_df, hui_slots_gq, prechui_df)
for check, (passed, detail) in linkage_checks.items():
    print(f"   [{'PASS' if passed else 'FAIL'}] {check:<40} {detail}")

# 'every person has a huid' is EXPECTED to fail under block-only matching -
# some blocks hold more person records than housing unit slots, a capacity
# floor, not a merge defect (Grays Harbor 2020 rs9876 reference: 85.30%
# assigned). The real gate is that every OTHER check passes.
structural = {k: v for k, v in linkage_checks.items()
              if k != 'every person has a huid'}
print()
print("All structural checks pass:", all(ok for ok, _ in structural.values()))
print("Note: 'every person has a huid' is expected to fail under block-only")
print("      matching. The unplaced are people in blocks holding more")
print("      residents than housing unit slots - a capacity floor, not a")
print("      defect in the merge.")

In [ ]:
# ==============================================================
# Step 5: Give each linked person their housing unit's building
#
# The HUA loop saved its full allocation file under its output folder. The
# current process_communities keeps only ['huid', bldg_uniqueid,
# placeNAME{yr}, 'huestimate', 'x', 'y'] from the allocation (ncoda_07i
# hua_cols), so the building linkage a person can inherit is the building id
# and coordinates; strctid/addrptid exist only in files written by older
# versions of the workflow and are picked up when present.
# ==============================================================
hua_version_text = 'v2-2-0'   # the version_text the HUA loop above passes
bldg_inv_id = communities[community_id]['building_inventory']['id']
bldg_uniqueid = communities[community_id]['building_inventory']['bldg_uniqueid']

# Current process_communities writes hua_{vt}_{community}_{vintage}_rs{seed}_
# {bldg_inv_id}.csv at the top of its output folder; older runs wrote
# ..._{bldg_inv_id}_rs{seed}.csv inside the community subfolder. Accept either.
hua_candidates = [
    (f"{hua_outputfolder}/hua_{hua_version_text}_{community_id}"
     f"_{intersect_vintage}_rs{intersect_seed}_{bldg_inv_id}.csv"),
    (f"{hua_outputfolder}/{community_id}/hua_{hua_version_text}_{community_id}"
     f"_{intersect_vintage}_{bldg_inv_id}_rs{intersect_seed}.csv"),
]
hua_full_path = next((p for p in hua_candidates if os.path.exists(p)), None)
if hua_full_path is None:
    raise FileNotFoundError(
        "No HUA file found - looked for:\n  " + "\n  ".join(hua_candidates) +
        f"\nRun the Housing Unit Allocation cell above for basevintage "
        f"{intersect_vintage} first.")
hua_full_df = pd.read_csv(hua_full_path, low_memory=False)
print(f"HUA file: {hua_full_path}")

# process_communities left-merges the allocation onto the housing unit
# inventory and keeps only rows with a huid, so every row here is a housing
# unit; a unit that matched no building carries the 'missing building id'
# sentinel and NaN coordinates. The notna filter is a no-op on current files
# and a guard against older ones.
hua_full_df = hua_full_df[hua_full_df['huid'].notna()]
assert hua_full_df['huid'].is_unique, "huid duplicated in the HUA file"

# The file holds the same housing units as the in-memory ensemble frame;
# subset (not equality) keeps the check tolerant of older files.
in_memory_huids = set(hua_hui_gdf_dict[intersect_vintage][intersect_seed]['huid'])
assert set(hua_full_df['huid']) <= in_memory_huids, \
    "HUA file holds huids the in-memory ensemble does not - wrong seed or vintage?"

yr = intersect_vintage[2:]
wanted_bldg_cols = ['strctid', 'addrptid', bldg_uniqueid, 'huestimate',
                    f'placeNAME{yr}', 'x', 'y']
present_bldg_cols = [c for c in wanted_bldg_cols if c in hua_full_df.columns]
absent_bldg_cols = [c for c in wanted_bldg_cols if c not in hua_full_df.columns]
if absent_bldg_cols:
    print(f"HUA file does not carry {absent_bldg_cols} - the current "
          "process_communities saves only the building id, place name, "
          "huestimate and coordinates. Continuing with what it has.")

# m:1 - every person in an assigned household inherits that unit's building.
# Persons with huid null or -999 match nothing and keep NaN attributes.
prechui_bldg_df = prechui_df.merge(hua_full_df[['huid'] + present_bldg_cols],
                                   on = 'huid', how = 'left', validate = 'm:1')

# Rebuild point geometry the same way process_communities does
prechui_gdf = gpd.GeoDataFrame(
    prechui_bldg_df,
    geometry = gpd.points_from_xy(prechui_bldg_df.x, prechui_bldg_df.y),
    crs = "EPSG:4326")

# The join must not add or lose people; coordinate gaps should be only the
# persons in units that matched no building in the allocation - reported, not
# asserted zero.
assert len(prechui_bldg_df) == len(prechui_df), \
    "building join changed the person count"
assigned = ~linkage.unassigned_mask(prechui_bldg_df['huid'])
no_xy = int((assigned & prechui_bldg_df['x'].isna()).sum())
print(f"Persons                              : {len(prechui_bldg_df):,}")
print(f"Assigned to a housing unit           : {int(assigned.sum()):,}")
print(f"Assigned persons without coordinates : {no_xy:,} "
      "(units that matched no building in the allocation - expected small)")

prechui_bldg_path = (f"{prechui_outputfolders['top']}/"
                     f"prechui_bldg_{prechui_version_text}_{state_county}"
                     f"_{intersect_vintage}_rs{intersect_seed}.csv")
prechui_gdf.to_csv(prechui_bldg_path, index = False)
print(f"Saved: {prechui_bldg_path}")

In [ ]:
# ==============================================================
# Step 6: Validation against Census tables the merge was NOT fitted to
#
# Agreement with the merge's own inputs - householder age band, sex, race,
# ethnicity - is guaranteed by construction. PCT5 and PCT6 describe whether a
# household CONTAINS someone above an age threshold, which is decided
# entirely by the linkage. Both tables live in the 2020 DHC; api.census.gov
# has no 2010 dec/dhc endpoint, so the check runs for 2020 only.
# ==============================================================
import time

if intersect_vintage == '2020':
    api_key = os.environ['CENSUS_API_KEY']
    state, county = state_county[0:2], state_county[2:5]

    census_counts = {}
    for table, spec in prechui_workflow_functions.outofsample_tables.items():
        variables = [v for k, v in spec.items()
                     if isinstance(v, str) and v.startswith(table)]
        url = (f"https://api.census.gov/data/{intersect_vintage}/dec/dhc"
               f"?get={','.join(variables)}"
               f"&for=county:{county}&in=state:{state}&key={api_key}")
        # A bad response here is not JSON: over-limit throttling and key
        # errors come back as text or HTML - historically with status 200 -
        # and .json() then fails far from the cause. Retry transients with
        # backoff; if it still fails, show the body, not a decode error.
        payload = None
        for attempt in range(4):
            response = requests.get(url, timeout = 120,
                                    allow_redirects = False)
            content_type = response.headers.get('content-type', '')
            if response.status_code == 200 and 'json' in content_type:
                payload = response.json()
                break
            print(f"{table} attempt {attempt + 1}: status "
                  f"{response.status_code}, {content_type or 'no type'} - "
                  f"retrying in {2 ** attempt}s")
            time.sleep(2 ** attempt)
        if payload is None:
            raise RuntimeError(
                f"Census API refused the {table} request after 4 tries: "
                f"status {response.status_code}, body starts "
                f"{response.text[:200]!r}")
        header, values = payload[0], payload[1]
        census_counts[table] = {v: int(values[header.index(v)])
                                for v in variables}

    validation_df = linkage.validate_against_census(prechui_df, census_counts)
    display(validation_df)

    print("\nIncomplete households bias every comparison downward, so the")
    print("'complete' rows are the fair test. One-person households are")
    print("trivially complete and skew old, so the size-class rows remove")
    print("that confound.")
else:
    print("Out-of-sample validation skipped: PCT5/PCT6 come from the 2020")
    print(f"DHC, and api.census.gov has no {intersect_vintage} dec/dhc")
    print("endpoint.")

### Interpreting the unassigned share

`every person has a huid` fails by design under block-only matching. The two inventories
distribute people to blocks differently, so some blocks hold more person records than
they have housing-unit slots - those people are structurally unplaceable *within their
block*. That is a capacity floor, not a defect in the merge: the Grays Harbor 2020
`rs9876` reference run placed 85.30% of 75,636 persons with every other structural check
passing.

Widening `geo_levels` to `['Block','Tract','County']` in the single call barely helps age
fidelity, because geography nests outside the rounds - the age-blind catch-all fires at
Block before the age-preserving rounds ever see the wider pool. If placement rate matters
more than block fidelity, the correct escalation is `linkage.merge_prec_to_hui_staged(...)`,
which runs every age-preserving round at the wider geographies before any round gives age
up.

One payload note: the Person Record cell above merges only the overall disability
status (`B18101`), so that is the disability payload the PRECHUI output carries. The six
difficulty tables (`B18102`-`B18107`: hearing, vision, cognitive, ambulatory, self-care,
independent living) currently live in the standalone `ncoda_07kv1_PREC_Disability.ipynb`;
integrating them into `generate_prec_functions` is upstream Issue #140.

One building-payload caveat: the current HUA workflow saves only the building id
(`fd_id_bid`), place name, `huestimate` and coordinates into its allocation file
(`hua_cols` in `ncoda_07i_process_communities.py`), so `strctid` and `addrptid` appear in
the building join's output only when the HUA file on disk was written by an older, richer
version of the workflow. Step 5 reports which columns it found.

## Explore and Validate Housing Unit Allocation


### Look at population characteristics and compare to US Census

In [ ]:
focalplace = communities[community_id]['community_name']
print(focalplace, focalplace, countyname, countyfips)

In [ ]:
hua_gdf = hua_hui_gdf_dict['2010'][9876]

In [ ]:
PopResultsTable.pop_results_table(
                  input_df = hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(
                  input_df = hua_hui_gdf_dict['2020'][9876], 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = '2020 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(hua_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
hua_gdf.head()

In [ ]:
# Show population counts by placeNAME10 - weight by numprec
hua_gdf.groupby('placeNAME10').size()

In [ ]:
focalplace

In [ ]:
# set dataframe for focal place
focalplace_hua_gdf =  hua_gdf.loc[hua_gdf['placeNAME10'] == 'Seaside'].copy(deep=True)

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
# set dataframe for focal place
hua_gdf_2020 = hua_hui_gdf_dict['2020'][9876]
focalplace_hua_gdf_2020 =  hua_gdf_2020.loc[hua_gdf_2020['placeNAME20'] == 'Seaside'].copy(deep=True)

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf_2020, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2020 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2010 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf_2020, 
                   who = "Median Household Income", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = '2020 (9876)',
                  row_index = "Race Ethnicity",
                  col_index = 'Tenure Status')

In [ ]:
hua_gdf['fd_id_bid'].describe()

In [ ]:
bldg_uniqueid = 'fd_id_bid'
# add category for missing building id
buildingdata_conditions = {'cat_var' : {'variable_label' : 'Building Data Availability',
                         'notes' : 'Does Housing Unit have building data?'},
              'condition_list' : {
                1 : {'condition': f"(df['{bldg_uniqueid}'] == 'missing building id')", 'value_label': "0 Missing Building Data"},
                2 : {'condition': f"(df['{bldg_uniqueid}'] != 'missing building id')", 'value_label': "1 Building Data Available"}}
            }
hua_gdf = add_label_cat_conditions_df(hua_gdf, conditions = buildingdata_conditions)

In [ ]:
PopResultsTable.pop_results_table(hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = countyname,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

In [ ]:
focalplace_hua_gdf = add_label_cat_conditions_df(focalplace_hua_gdf, conditions = buildingdata_conditions)

PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Race, Ethnicity",
                  where = focalplace,
                  when = "2010",
                  row_index = "Race Ethnicity",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Tenure Status",
                  where = focalplace,
                  when = "2010",
                  row_index = "Tenure Status",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

In [ ]:
PopResultsTable.pop_results_table(focalplace_hua_gdf, 
                  who = "Total Population by Households", 
                  what = "by Income Groups",
                  where = focalplace,
                  when = "2010",
                  row_index = "Household Income Group",
                  col_index = 'Building Data Availability_str',
                  row_percent = '0 Missing Building Data')

#### Validate the Housing Unit Allocation has worked
Notice that the population count totals for the community
should match (pretty closely) data collected for the 2010 Decennial Census.
This can be confirmed by going to data.census.gov

In [ ]:
print("Total Population by Race and Ethnicity:")
print(f"https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=DECENNIALSF12010.P5")

print("Median Income by Race and Ethnicity:")
print(f"All Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013")
print(f"Black Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013B")
print(f"White, not Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013H")
print(f"Hispanic Households: https://data.census.gov/cedsci/table?g=050XX00US{countyfips}&tid=ACSDT5Y2012.B19013I")

Differences in the housing unit allocation and the Census count may be due to differences between political boundaries and the building inventory. See Rosenheim et al 2019 for more details.

The housing unit allocation, plus the building results will become the input for the social science models such as the population dislocation model.